# Decision Tree

Decision Tree can be used for both
- Classification (0 or 1)
- Regression (predict number)

# Gini
We first split the dataset into left and right child nodes. Then we calculate the Gini impurity of each child. Since one child may contain more samples than the other, we multiply each child's Gini by its proportion of the total samples. Finally, we add these weighted impurities together. The split with the lowest Weighted Gini is chosen because it produces the purest child nodes overall.

In [688]:
import numpy as np
import pandas as pd

# Dataset

In [689]:
df = pd.DataFrame({
    "Age": [22,25,28,35,40,45,50,55],
    "Income": [30,35,40,50,60,70,80,90],
    "Buy": [0,0,1,1,1,0,1,1]
})

df

,Age,Income,Buy
0,22,30,0
1,25,35,0
2,28,40,1
3,35,50,1
4,40,60,1
5,45,70,0
6,50,80,1
7,55,90,1


# Split the dataset into X and Y

In [690]:
X = np.array([df["Age"], df["Income"]]).T

y = np.array(df["Buy"])

print("X")
print(X)

print("y")
print(y)

X
[[22 30]
 [25 35]
 [28 40]
 [35 50]
 [40 60]
 [45 70]
 [50 80]
 [55 90]]
y
[0 0 1 1 1 0 1 1]


In [691]:
def gini(y):

    if len(y) == 0:
        return 0
    
    # return_counts-> gives how much time it reapets (1, 1, 2, 2, 2) -> class = (1, 2), count = (2, 3)
    values, counts = np.unique(y, return_counts = True)

    probs = counts/np.sum(counts)

    return 1 - np.sum(probs ** 2)

In [692]:
def split(X, y, feature, threshold):
    
    # find l and r based on threshold
    
    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    return X[left_mask], X[right_mask], y[left_mask], y[right_mask]

In [693]:
def get_threshold(thresholds, y):

    # thresholds = np.sort(np.unique(thresholds))
    index = np.argsort(thresholds)

    thresholds = thresholds[index]
    y = y[index]

    new_thresholds = []

    for i in range(len(thresholds)-1):

        # optimization 
        # 22 -> 0
        # 22 -> 1
        # 25 -> 1
        # But then the split becomes Age <= 22 so to stop this we added thresholds not same
        
        # if thresholds[i] != thresholds[i+1] and y[i] != y[i+1]:
        if y[i] != y[i+1]:
            new_thresholds.append((thresholds[i] + thresholds[i+1])/2)

    # print(thresholds)
    # print(y)
    # print(new_thresholds)
    
    return np.array(new_thresholds)

In [694]:
def best_split(X, y):
    best_gini = float("inf")
    best_feature = None
    best_threshold = None

    # count of features(column)
    n_samples, n_features = X.shape

    # iter each feature
    for feature in range(n_features):
        
        # row based on feature
        thresholds = X[:, feature]

        thresholds = get_threshold(thresholds, y)

        # iter each row
        for threshold in thresholds:

            XL, XR, yL, yR = split(X, y, feature, threshold)            

            # skip if yL or yR is empty
            if len(yL) == 0 or len(yR) == 0:
                continue

            # find gini
            gini_l = gini(yL)
            gini_r = gini(yR)

            # combine gini l and r 
            weighted_gini = (len(yL) / n_samples) * gini_l + (len(yR) / n_samples) * gini_r

            # finding best gini
            if best_gini > weighted_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold

            # print(f"feature : {feature}, threshold : {threshold}")
            
    #         print("XL")
    #         print(XL)
    #         print("XR")
    #         print(XR)
    #         print("yL")
    #         print(yL)
    #         print("yR")
    #         print(yR)

    #         print("GINI")
    #         print(gini_l)
    #         print(gini_r)

    
    # print("BEST")
    # print("best gini : ", best_gini)
    # print("best feature : ", best_feature)
    # print("best threshold : ", best_threshold)

    return best_feature, best_threshold

In [695]:
def build_tree(X, y, depth = 0, max_depth = 3):

    # y = [0 0 0] label 0 or [1 1 1] label 1
    if len(np.unique(y)) == 1:
        return {'label' : y[0]}
    
    # depth >= max_depth find max value count and label it.
    if depth >= max_depth:
        # 3 >= 3 -> y = [0 1 1 1] return label 1
        values, counts = np.unique(y, return_counts = True)
        return {"label" : values[np.argmax(counts)]}
    

    feature, threshold = best_split(X, y)

    # best_split give None as threshold and feature
    if feature is None:
        # 3 >= 3 -> y = [0 0 1] return label 0
        values, counts = np.unique(y, return_counts = True)
        return {"label" : values[np.argmax(counts)]}

    XL, XR, yL, yR = split(X, y, feature, threshold)

    return {"feature": feature, "threshold": threshold, "left": build_tree(XL, yL, depth+1, max_depth), "right": build_tree(XR, yR, depth+1, max_depth)}

In [696]:
tree = build_tree(X, y, max_depth = 5)
tree

{'feature': 0,
 'threshold': np.float64(26.5),
 'left': {'label': np.int64(0)},
 'right': {'feature': 0,
  'threshold': np.float64(42.5),
  'left': {'label': np.int64(1)},
  'right': {'feature': 0,
   'threshold': np.float64(47.5),
   'left': {'label': np.int64(0)},
   'right': {'label': np.int64(1)}}}}

# Visualizer

In [697]:
def print_tree(tree, feature_names, indent="", branch="Root"):
    
    # Leaf Node
    if "label" in tree:
        print(f"{indent}{branch} -> Predict: {tree['label']}")
        return

    # Decision Node
    feature = feature_names[tree["feature"]]
    threshold = tree["threshold"]

    print(f"{indent}{branch} -> {feature} <= {threshold}")

    # Left Child
    print_tree(
        tree["left"],
        feature_names,
        indent + "│   ",
        "True"
    )

    # Right Child
    print_tree(
        tree["right"],
        feature_names,
        indent + "│   ",
        "False"
    )

print_tree(tree, df.columns)

Root -> Age <= 26.5
│   True -> Predict: 0
│   False -> Age <= 42.5
│   │   True -> Predict: 1
│   │   False -> Age <= 47.5
│   │   │   True -> Predict: 0
│   │   │   False -> Predict: 1


In [698]:
def predict(x, tree):
    
    if "label" in tree:
        return tree["label"]
    
    if x[tree["feature"]] <= tree["threshold"]:
        return predict(x, tree["left"])
    else:
        return predict(x, tree["right"])

predict((1, 1), tree)

np.int64(0)

# Problem 2

In [699]:
X = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
])

y = np.array([
    0,
    1,
    1,
    0
])


tree = build_tree(X, y, max_depth = 100)
tree

{'feature': 0,
 'threshold': np.float64(0.0),
 'left': {'feature': 1,
  'threshold': np.float64(0.5),
  'left': {'label': np.int64(0)},
  'right': {'label': np.int64(1)}},
 'right': {'feature': 1,
  'threshold': np.float64(0.5),
  'left': {'label': np.int64(1)},
  'right': {'label': np.int64(0)}}}

In [700]:
# type(df.columns)

print_tree(tree, ["x", "y"])

Root -> x <= 0.0
│   True -> y <= 0.5
│   │   True -> Predict: 0
│   │   False -> Predict: 1
│   False -> y <= 0.5
│   │   True -> Predict: 1
│   │   False -> Predict: 0


In [701]:
def predict(x, tree):
    
    if "label" in tree:
        return tree["label"]
    
    if x[tree["feature"]] <= tree["threshold"]:
        return predict(x, tree["left"])
    else:
        return predict(x, tree["right"])

predict((0, 0), tree)

np.int64(0)